<a href="https://colab.research.google.com/github/ayushniet01/NLP-Program-using-Python-NLTK-and-spaCy/blob/main/Untitled31.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Automatic Document Classification and Separation using ML

This notebook outlines a pipeline for classifying and separating documents using machine learning techniques.

## Pipeline Overview
1. **Setup & Dependencies**: Install and import required libraries.
2. **Data Loading & Preprocessing**: Load text, PDF, or image documents and clean the data.
3. **Feature Extraction**: Convert text data into numerical features (using TF-IDF, Word Embeddings, or Hugging Face Transformers).
4. **Model Training**: Train classification models (e.g., Logistic Regression, Random Forest, or Deep Learning models).
5. **Evaluation**: Evaluate performance using accuracy, precision, recall, and F1-score.
6. **Document Separation**: Logic to split multi-page documents based on classification boundaries.

In [ ]:
# Install necessary libraries
!pip install -q scikit-learn pandas numpy matplotlib seaborn

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, confusion_matrix

print("Setup complete. Libraries imported successfully!")

## 2. Mock Data Generation & Preprocessing

To demonstrate document classification and boundary separation, we will generate a mock dataset of document pages belonging to three categories:
* **Invoice**
* **Resume**
* **Scientific Paper**

In [ ]:
# Generate mock document data
data = [
    # Invoices
    ("Invoice #1024. Total balance due: $1,250. Please send payments to Acme Corp.", "Invoice"),
    ("Purchase order invoice invoice. Tax invoice amount $300, payment terms net 30.", "Invoice"),
    ("Billing invoice for consulting services. Quantity 10 hours rate $100.", "Invoice"),
    # Resumes
    ("Experienced software developer resume. Skills: Python, Machine Learning, SQL, and Java.", "Resume"),
    ("Professional CV of Jane Doe. Education: BS in Computer Science. Work experience at Tech Inc.", "Resume"),
    ("Data Scientist resume. Proficient in statistical analysis, PyTorch, and NLP modeling.", "Resume"),
    # Scientific Papers
    ("Abstract: We present a novel deep neural network architecture for document segmentation tasks.", "Scientific Paper"),
    ("Introduction. Recent advances in transformer models have improved natural language processing benchmarks.", "Scientific Paper"),
    ("References. Smith et al. (2021). Journal of Machine Learning Research, 12(3), 45-67.", "Scientific Paper")
]

# Create a DataFrame
df = pd.DataFrame(data, columns=["text", "label"])
print("Dataset shape:", df.shape)
print("\nSample data:")
print(df.head())

## 3. Feature Extraction & Model Training

In this section, we will convert the text documents into numerical representations using **TF-IDF Vectorization**, split our data into training and validation sets, and train a **Logistic Regression** classifier.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Split features and labels
X = df["text"]
y = df["label"]

# Vectorize text data using TF-IDF
vectorizer = TfidfVectorizer(stop_words='english')
X_tfidf = vectorizer.fit_transform(X)

# Train-test split (using a small split since dataset is small)
X_train, X_test, y_train, y_test = train_test_split(X_tfidf, y, test_size=0.33, random_state=42, stratify=y)

# Train Classifier
model = LogisticRegression()
model.fit(X_train, y_train)

print("Model trained successfully on TF-IDF features!")

## 4. Document Separation Logic

In a real-world scenario, documents arrive as a continuous stream of mixed pages (e.g., a multi-page PDF combining invoices, resumes, and articles).

We will use our trained model to classify each incoming page sequentially and separate them into individual files when a transition boundary is detected.

In [ ]:
# Simulate a sequential stream of document pages
mixed_document_stream = [
    "Invoice for client design project. Balance due $500.",
    "Purchase order invoice for software licenses. Total $2000.",
    "Senior Data Engineer CV. Experienced in Apache Spark, Scala, and Python.",
    "Resume for Frontend Developer. Expertise in React, CSS, and HTML.",
    "We present an analysis on the efficiency of modern gradient descent methods.",
    "References and citations list for the optimization paper."
]

# Classify pages and identify separation boundaries
def separate_document_stream(pages):
    separated_docs = []
    current_doc = []
    current_type = None

    # Transform texts with the trained vectorizer
    features = vectorizer.transform(pages)
    predictions = model.predict(features)

    for idx, (page_text, pred_label) in enumerate(zip(pages, predictions)):
        print(f"Page {idx+1} classified as: {pred_label}")

        # Boundary detection: Start a new document if type changes
        if current_type is not None and pred_label != current_type:
            separated_docs.append((current_type, current_doc))
            current_doc = []

        current_doc.append(page_text)
        current_type = pred_label

    # Append final document
    if current_doc:
        separated_docs.append((current_type, current_doc))

    return separated_docs

# Run separation
results = separate_document_stream(mixed_document_stream)

print(f"\n--- Separation Completed: Split into {len(results)} distinct documents ---")
for i, (doc_type, pages) in enumerate(results, 1):
    print(f"\nDocument {i} [{doc_type}]: {len(pages)} pages")
    for idx, p in enumerate(pages, 1):
        print(f"  -> Page {idx}: {p[:60]}...")

### Expected Output of Running the Stream Separation:

```text
Page 1 classified as: Invoice
Page 2 classified as: Invoice
Page 3 classified as: Resume
Page 4 classified as: Resume
Page 5 classified as: Scientific Paper
Page 6 classified as: Scientific Paper

--- Separation Completed: Split into 3 distinct documents ---

Document 1 [Invoice]: 2 pages
  -> Page 1: Invoice for client design project. Balance due $500....
  -> Page 2: Purchase order invoice for software licenses. Total $2000....

Document 2 [Resume]: 2 pages
  -> Page 1: Senior Data Engineer CV. Experienced in Apache Spark, Scal...
  -> Page 2: Resume for Frontend Developer. Expertise in React, CSS, an...

Document 3 [Scientific Paper]: 2 pages
  -> Page 1: We present an analysis on the efficiency of modern gradient...
  -> Page 2: References and citations list for the optimization paper....
```